# Saving Models

In [1]:
from ngboost import NGBRegressor

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

X, Y = load_diabetes(return_X_y=True)
X_reg_train, X_reg_test, Y_reg_train, Y_reg_test = train_test_split(X, Y, test_size=0.2)

Saving ngboost models is easy with the `pickle` package:

In [2]:
ngb = NGBRegressor().fit(X_reg_train, Y_reg_train)

[iter 0] loss=5.7638 val_loss=0.0000 scale=1.0000 norm=65.5867
[iter 100] loss=5.3738 val_loss=0.0000 scale=2.0000 norm=86.9325
[iter 200] loss=5.1450 val_loss=0.0000 scale=2.0000 norm=71.5014
[iter 300] loss=5.0053 val_loss=0.0000 scale=1.0000 norm=32.1308
[iter 400] loss=4.8992 val_loss=0.0000 scale=1.0000 norm=30.0707


In [3]:
import pickle
from pathlib import Path

file_path = Path('ngbtest.p')

with file_path.open("wb") as f:
    pickle.dump(ngb, f)

In [4]:
with file_path.open("rb") as f:
    ngb_unpickled = pickle.load(f)

In [5]:
Y_preds = ngb_unpickled.predict(X_reg_test)
Y_dists = ngb_unpickled.pred_dist(X_reg_test)

Y_dists[0:5].params

{'loc': array([106.47944729, 164.87039675, 213.2708231 , 167.96627052,
        159.96009392]),
 'scale': array([39.69338954, 37.66577362, 24.97050121, 46.87232726, 41.10389475])}

To avoid pickle, you can use [skops](https://skops.readthedocs.io), which does not run arbitrary code when it loads a file. Compress the file to keep it small, and check the list of types before you trust them:

In [6]:
from zipfile import ZIP_DEFLATED

import skops.io as sio

sio.dump(ngb, "ngbtest.skops", compression=ZIP_DEFLATED)

unknown_types = sio.get_untrusted_types(file="ngbtest.skops")
unknown_types

['ngboost.api.NGBRegressor',
 'ngboost.distns.normal.Normal',
 'ngboost.scores.LogScore',
 'sklearn.tree._tree.Tree']

In [7]:
ngb_skops = sio.load("ngbtest.skops", trusted=unknown_types)

ngb_skops.pred_dist(X_reg_test)[0:5].params

{'loc': array([106.47944729, 164.87039675, 213.2708231 , 167.96627052,
        159.96009392]),
 'scale': array([39.69338954, 37.66577362, 24.97050121, 46.87232726, 41.10389475])}